In [1]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from pathlib import Path
import pandas as pd
import seaborn as sns
import numpy as np
from tailnflows.utils import load_raw_data, load_experiment_output_data, get_data_path

In [2]:
# Load experiment results
experiment_output = load_experiment_output_data('insurance/2026-09-04-de-test')
# for k, v in experiment_output.items():
#     print(f"{k}: {v}")

# Put experiment results in dataframe
rows = []
for model, run_datas in experiment_output.items():
    for rd in run_datas:
        rows.append({
            'data': model.split('-')[0],
            **rd
        })

df = pd.DataFrame(rows)
print(df)

         data model  dim  seed  split   tst_nll   val_nll       sw2     w1_ht  \
0   insurance  mtaf    2    17      1  3.718051  3.598624  1.985516  1.224725   
1   insurance  gtaf    2    17      1  3.717925  3.598511  1.990547  1.223829   
2   insurance  gtaf    2     0      0  4.919598  4.745366  2.239118  2.666015   
3   insurance  mtaf    2     0      0  4.939118  4.734185  5.013905  3.092036   
4   insurance  mtaf    2    34      2  3.832367  3.903182  2.568981  2.201735   
5   insurance  mtaf    2    51      3  4.166951  4.264821  6.260466  2.179983   
6   insurance  gtaf    2    34      2  3.859440  3.915752  2.876698  2.334372   
7   insurance  gtaf    2    51      3  4.166839  4.264724  6.216942  2.177857   
8   insurance  mtaf    2    68      4  3.746459  3.708215  1.228375  0.831179   
9   insurance  gtaf    2    68      4  3.746355  3.708104  1.225838  0.830911   
10  insurance  gtaf    2    85      5  3.093992  3.288791  8.262325  3.524426   
11  insurance  mtaf    2    

In [3]:
grouper = ['model', 'depth']

# select only experiments with specific parameters
selector = np.ones(len(df), dtype=bool)
# selector &= df['batch_size'] == 100
# selector &= df['tail_bound'] == 2.5
# selector = np.logical_and(
#     df['batch_size'] == 100,
#     df['tail_bound'] == 2.5
# )
df = df[selector]

# aggregate nll results over splits
grouped = df.groupby(grouper)
agg = grouped[['val_nll', 'tst_nll', 'sw2', 'w1_ht', 'w1_lt', 'var99_ht', 'var99_lt', 'var995_ht', 'var995_lt', 'var999_ht', 'var999_lt']].agg(['mean', 'sem', 'std'])
print(agg)

# Select best test nll results and its std
best = agg['tst_nll'].sort_values(by='mean') # type: ignore
within = best.iloc[0]['mean'] + 1.0 * best.iloc[0]['std']

def highlight_by_mean(row, threshold=7):
    """
    Highlight the row if the 'Mean' value is below the given threshold.
    """
    if row['mean'] < threshold:
        # Highlight entire row in a light green
        return ['background-color: #dfffdf'] * len(row) # light-green
    else:
        # No highlight
        return ['background-color: white'] * len(row)

# Show sorted results
best.style.apply(
    highlight_by_mean, 
    axis=1, 
    threshold=within
)

              val_nll                       tst_nll                      \
                 mean       sem       std      mean       sem       std   
model depth                                                               
gtaf  1      4.091778  0.187626  0.593325  4.088795  0.201830  0.638241   
mtaf  1      4.095128  0.181746  0.574733  4.083036  0.203462  0.643403   

                  sw2                         w1_ht  ... var995_ht var995_lt  \
                 mean       sem       std      mean  ...       std      mean   
model depth                                          ...                       
gtaf  1      3.641145  0.690805  2.184517  1.979509  ...  5.868160  1.186519   
mtaf  1      3.495863  0.521702  1.649767  1.980947  ...  5.598931  0.887324   

                                 var999_ht                       var999_lt  \
                  sem       std       mean        sem        std      mean   
model depth                                                        

/home/vonbalnd/micromamba/envs/py312/lib/python3.12/site-packages/pandas/core/nanops.py:1744: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:820.)
  x = float(x)


,,mean,sem,std
model,depth,,,
mtaf,1,4.083036,0.203462,0.643403
gtaf,1,4.088795,0.201830,0.638241


In [ ]:
# Plot final test nll against final validation nll for all models (lower test nll is better)!

color_palette = plt.get_cmap('tab10', len(agg.index))  # Using a colormap

model_to_color = {model: color_palette(i) for i, model in enumerate(agg.index)}

selected_ix = []
for ix, group in grouped:
    plt.scatter(group.val_ll, group.tst_ll, color=model_to_color[ix], marker='.')
    # plt.scatter(ix[-2], group.tst_ll, c=model_to_color[ix])
    ix = (ix[0], int(ix[1]))
    selected_ix.append(ix)

plt.ylabel('tst')
plt.xlabel('val')
legend_elements = [Patch(facecolor=model_to_color[model], edgecolor='black', label=model) 
                   for model in selected_ix] # integer in the models legend represents depth parameter
plt.legend(handles=legend_elements, title="Models", loc='upper left', bbox_to_anchor=(1.05, 1))
plt.grid()
plt.show()

In [ ]:
# print evolution of train and val loss for all models
for label, runs in experiment_output.items():
    for run in runs:
        losses = load_raw_data(run['loss_path'])[label][
            run['loss_ix']
        ]

        plt.title(f'{label} lr={run["lr"]} | batch_size={run["batch_size"]}')
        l, = plt.plot(np.arange(len(losses['losses'])), losses['losses'], label='') # training losses
        plt.plot(losses['steps'], losses['vlosses'], linestyle='--', c=l.get_color()) # validation losses
        plt.xlabel('update steps')
    plt.grid()
    plt.show()

In [ ]:
# Inspect split data
data_source = "climate"
split = 0
tail_path = f'{get_data_path()}/splits/{data_source}/{split}'
if not Path(f"{tail_path}.p").is_file():
    raise Exception(
        f"Split data not present at {tail_path}.p, either configure "
        "TAILNFLOWS_DATA_DIR, or run `python experiments/density_estimation_real_data/generate_splits.py`"
    )

splits_and_tail = load_raw_data(tail_path)["experiment_data"][0]
print(splits_and_tail)

metadata = splits_and_tail["metadata"]
# print("Metadata: ", metadata)
dfs = metadata["dfs"]
pos_dfs = metadata["pos_dfs"]
neg_dfs = metadata["neg_dfs"]
seed = metadata["seed"]
if data_source != "climate": # mean and std split data are not available for climate
    mean = metadata["mean"]
    std = metadata["std"]

print(f"Split data for {data_source} split {split}:")

if data_source != "climate":
    print(f"Mean: {[round(float(m), 2) for m in mean]}")
    print("len(mean): ", len(mean))
    print(f"Std: {[round(float(s), 2) for s in std]}")
print(f"Seed: {seed}")
print(f"dfs: {[round(df, 2) for df in dfs]}")
print(f"pos_dfs: {[round(df, 2) for df in pos_dfs]}")
print(f"neg_dfs: {[round(df, 2) for df in neg_dfs]}")

avg_df, std_df = np.mean(dfs), np.std(dfs)
avg_pos_df, std_pos_df = np.mean(pos_dfs), np.std(pos_dfs)
avg_neg_df, std_neg_df = np.mean(neg_dfs), np.std(neg_dfs)

print(f"Average df: {avg_df:.2f} ({std_df:.2f})")
print(f"Average pos df: {avg_pos_df:.2f} ({std_pos_df:.2f})")
print(f"Average neg df: {avg_neg_df:.2f} ({std_neg_df:.2f})")

In [ ]:
# Add test NLL/dim as column
df['test_neg_ll_per_dim'] = df['tst_ll'] / df['dim']
df.head()

In [ ]:
# Add a readable_name column
name_map = {
    'normal': 'Normal (baseline)',
    'ttf': 'TTF',
    'ttf_fix': 'TTF (fix)',
    'ttf_hdonly': 'TTF (hd_only)',
    'ttf_lin': 'TTF (lin)',
    'ttf_qua': 'TTF (qua)',
    'ttf_erfi': 'TTF (erfi)',
    'mtaf': 'mTAF',
    'gtaf': 'gTAF',
}
df['readable_name'] = df['model'].apply(lambda x: name_map.get(x, x))

# # Count non-NA cells for each column
df.groupby('readable_name').count()

In [ ]:
# Plot test nll per dimension for experiment data of specified dimensionality

target_dims = [2]
fig, axarr = plt.subplots(1, len(target_dims), figsize=(18, 6), tight_layout=True, sharey=True)
if len(target_dims) == 1:
  axarr = [axarr]

sns.set_style("whitegrid")
sns.set_context("paper", font_scale=2.)

for i, dim in enumerate(target_dims):
  j = 1

  wanted_data = df[df['dim'] == dim]

  sns.boxplot(
      data=wanted_data,
      x='test_neg_ll_per_dim',
      y='readable_name',
      order=name_map.values(),
      orient='h',
      ax=axarr[i],
  )

  axarr[i].axhline(1.5, linestyle='--', c='black')
  axarr[i].set_title(f'dim={dim}')
  axarr[i].set_xlabel('')
  axarr[i].set_ylabel('Model')

  fig.add_subplot(111, frameon=False)
  plt.xlabel('Test Negative Log-Likelihood per Dimension')
  plt.tick_params(labelcolor='none', which='both', top=False, bottom=False, left=False, right=False)
  plt.tight_layout()
  # plt.savefig(f"./real_sp500.png")